# new_pipeline_loo – smoke test
Checks the ElasticNet LOO pipeline on the first proteins of `data/protein_list.txt`.

Needs (inside `03. elasticNet/data/`): `transcriptomics_filtered.joblib`, `proteomics_matched.joblib`, `protein_list.txt`.
Writes to `../data/loo/results_smoke/` (never touches the real `results_<norm>/` folders).

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "new_smoke"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import sys, subprocess, shlex
import pandas as pd
from joblib import load

ROOT = Path('.').resolve()
DATA_DIR, SRC_DIR = ROOT.parent / 'data', ROOT / 'src'   # single 03. elasticNet/data folder
RESULTS_DIR = DATA_DIR / 'loo' / 'results_smoke'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RNA_FILE = 'transcriptomics_filtered.joblib'          # same file the CLI uses by default
transcript_path = DATA_DIR / RNA_FILE
protein_path = DATA_DIR / 'proteomics_matched.joblib'
protein_list_path = DATA_DIR / 'protein_list.txt'
N_TEST = 2                                            # proteins to test
for p in (transcript_path, protein_path, protein_list_path):
    assert p.exists(), f'Missing {p}'
print('All inputs found in', DATA_DIR)

In [ ]:
X, Y = load(transcript_path), load(protein_path)
proteins = [p.strip() for p in protein_list_path.read_text().splitlines() if p.strip() and not p.startswith('#')]
test_proteins = proteins[:N_TEST]
print('X', X.shape, '| Y', Y.shape, '| testing', test_proteins)

## 1. In-process call (best-only, z-score)

In [ ]:
sys.path.insert(0, str(SRC_DIR))
from new_functions import run_protein_group
param_grid = {'alpha': [0.1, 0.5, 1.0], 'l1_ratio': [0.1, 0.5, 0.9]}
out_best = run_protein_group(proteins=test_proteins, pipeline='fixed_percentile', X=X, Y=Y,
                             param_grid=param_grid, percentile=5, n_splits=3, save_only_best=True,
                             output_dir=str(RESULTS_DIR / 'inprocess'), zscore=True, fit_intercept=False)
print(out_best[['protein', 'best_sample', 'best_abs_error', 'mse_all_others']])

## 2. CLI call, exactly as `new_run_enet.sh` runs it (`--full_output`)

In [ ]:
for norm in ['zscore', 'binning']:
    out_dir = RESULTS_DIR / f'results_{norm}'
    cmd = [sys.executable, str(SRC_DIR / 'new_pipeline.py'), '--start_idx', '0', '--group_size', str(N_TEST),
           '--pipeline', 'fixed_percentile', '--normalization', norm, '--percentile', '5', '--n_splits', '3',
           '--n_bins', '10', '--full_output', '--data_dir', str(DATA_DIR), '--rna_file', RNA_FILE,
           '--results_dir', str(out_dir), '--protein_list', str(protein_list_path)]
    print('CMD:', ' '.join(shlex.quote(c) for c in cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-600:]); assert r.returncode == 0, r.stderr[-2000:]

In [ ]:
for norm in ['zscore', 'binning']:
    out_dir = RESULTS_DIR / f'results_{norm}'
    for p in test_proteins:
        files = {k: sorted(out_dir.glob(f'{p}_{k}.*')) for k in ('best', 'LOO')}
        assert files['best'] and files['LOO'], f'missing outputs for {p} in {out_dir}'
    batch = sorted(out_dir.glob('batch_*'))
    print(norm, '->', [f.name for f in sorted(out_dir.iterdir())])
    loo = pd.read_parquet(sorted(out_dir.glob('*_LOO.parquet'))[0])
    print('LOO columns:', list(loo.columns), '| rows:', len(loo))